[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-09-background-tasks-middleware.ipynb#scrollTo=aa1122bb)

---
# Day 9 · Background Tasks, Middleware, and CORS
**certified-journeys / fastapi-certified** · Platform Features

> **Goal for today:** Add fire-and-forget background tasks to routes, write custom request-timing middleware, configure CORS for a frontend origin, and add GZip compression — all verified through `TestClient` response inspection.


In [ ]:
%pip install -q fastapi httpx


---
## Background tasks vs middleware vs CORS — overview

These three features solve different concerns at different levels of the request lifecycle:

```
Incoming request
  │
  ▼
┌─────────────────────────────────────────────┐
│  CORS middleware         (cross-origin check) │
│  GZip middleware         (decompress/compress)│
│  Timing middleware       (measure + log)      │
│  Route handler           (your logic)         │
│  BackgroundTasks         (after response sent) │
└─────────────────────────────────────────────┘
  │
  ▼
Response sent to client
  (BackgroundTasks run here, after the response is already sent)
```

| Feature | Use case | When it runs |
|---|---|---|
| `BackgroundTasks` | Email, audit log, cache warm | **After** response |
| Custom middleware | Timing, logging, auth headers | **Around** every request |
| `CORSMiddleware` | Allow browser cross-origin requests | **Before** route |
| `GZipMiddleware` | Compress large responses | **After** route, before send |


---
## Step 1 · `BackgroundTasks` — fire-and-forget

FastAPI's `BackgroundTasks` lets you schedule a function to run **after the HTTP response is already sent**. The client receives the response immediately — the task runs in the background.

Add `background_tasks: BackgroundTasks` as a route parameter — FastAPI injects it automatically (no `Depends` needed).

| Method | What it does |
|---|---|
| `background_tasks.add_task(fn, *args, **kwargs)` | Schedules `fn(*args, **kwargs)` to run after the response |
| Multiple `add_task` calls | Tasks run in the order added |


In [ ]:
import time
from fastapi import FastAPI, BackgroundTasks
from fastapi.testclient import TestClient

app = FastAPI(title='Background Tasks Demo')

# Shared log so we can inspect what background tasks ran in tests
task_log: list[str] = []


def send_welcome_email(email: str, username: str) -> None:
    """
    Simulates sending a welcome email.
    In production: call your email provider SDK here.
    This function runs AFTER the HTTP response has been sent.
    """
    # Simulate a slow operation (e.g., SMTP round-trip)
    time.sleep(0.05)
    message = f'Welcome email sent to {email} for user {username}'
    task_log.append(message)
    print('[BACKGROUND]', message)  # visible in server logs, not in response


def log_signup(username: str) -> None:
    """Simulates writing to an audit log."""
    message = f'Audit: new signup for {username}'
    task_log.append(message)
    print('[BACKGROUND]', message)


@app.post('/register')
def register_user(username: str, email: str, background_tasks: BackgroundTasks):
    """
    Registers a user and schedules background tasks.
    The response returns immediately — tasks run after.
    """
    # Schedule tasks — they run after this function returns
    background_tasks.add_task(send_welcome_email, email=email, username=username)
    background_tasks.add_task(log_signup, username=username)

    # This response is sent immediately — tasks haven't run yet
    return {'status': 'registered', 'username': username}


client = TestClient(app)

# Record time before and after the request
task_log.clear()
start = time.perf_counter()
resp = client.post('/register?username=alice&email=alice@example.com')
elapsed = time.perf_counter() - start

print('Response status:', resp.status_code)
print('Response body:  ', resp.json())
print(f'Request elapsed: {elapsed*1000:.1f}ms')
print()
print('Task log (ran during TestClient request):')  
for entry in task_log:
    print(' •', entry)


**What just happened?**
- The response `{"status": "registered"}` was returned **before** the background tasks completed.
- In `TestClient`, background tasks run synchronously within the request context (by default) — in a real server they run after the response is dispatched to the network socket.
- Both tasks ran in the order they were added: `send_welcome_email` → `log_signup`.


---
## Step 2 · BackgroundTasks via dependency injection

You can also inject `BackgroundTasks` into a **dependency** function — this lets utility functions schedule their own background work without the route needing to know.


In [ ]:
from fastapi import Depends

notification_log: list[str] = []


def get_notification_service(background_tasks: BackgroundTasks):
    """
    A dependency that wraps BackgroundTasks.
    The dependency itself schedules a startup notification.
    """
    def notify(message: str):
        notification_log.append(f'[notify] {message}')
        print('[BACKGROUND notify]', message)

    # Return a callable that routes can use to fire notifications
    def send(message: str):
        background_tasks.add_task(notify, message)

    return send


@app.post('/publish')
def publish_post(
    title: str,
    notify=Depends(get_notification_service),
):
    """Publishes a post and notifies subscribers in the background."""
    notify(f'New post published: {title}')  # schedules via dependency
    return {'published': title}


notification_log.clear()
resp = client.post('/publish?title=Hello+World')
print('Response:', resp.json())
print('Notifications fired:', notification_log)


**What just happened?**
- `BackgroundTasks` can be injected into **any dependency** in the chain — not just route functions.
- The dependency returns a `send` callable; the route uses it without importing or knowing about `BackgroundTasks` directly.
- This pattern is useful for notification services, audit loggers, and event emitters.


---
## Step 3 · Custom middleware with `@app.middleware("http")`

Middleware runs around **every** request. The `@app.middleware('http')` decorator is the simplest way to add custom behavior.

Structure:
```python
@app.middleware('http')
async def my_middleware(request: Request, call_next):
    # before: inspect/modify the request
    response = await call_next(request)  # call the route handler
    # after:  inspect/modify the response
    return response
```

The `call_next` function calls the next middleware or the route handler. Always `await` it and always return its response.


In [ ]:
import time as _time
from fastapi import Request
from fastapi.responses import Response

# Fresh app so we can add middleware cleanly
app2 = FastAPI(title='Middleware Demo')

# Store timing data for inspection in tests
timing_log: list[dict] = []


@app2.middleware('http')
async def add_process_time_header(request: Request, call_next) -> Response:
    """
    Measures how long each request takes and adds an X-Process-Time header
    to every response. Also logs to timing_log for test inspection.
    """
    start_time = _time.perf_counter()

    # call_next passes the request down the chain to the route handler
    response = await call_next(request)

    process_time = _time.perf_counter() - start_time
    ms = round(process_time * 1000, 3)

    # Add custom header to the response
    response.headers['X-Process-Time'] = f'{ms}ms'

    timing_log.append({
        'method': request.method,
        'path':   str(request.url.path),
        'status': response.status_code,
        'ms':     ms,
    })

    return response


@app2.get('/hello')
def hello(name: str = 'World'):
    return {'message': f'Hello, {name}!'}


@app2.get('/slow')
async def slow_endpoint():
    import asyncio
    await asyncio.sleep(0.05)  # simulate a slow DB query
    return {'status': 'done'}


client2 = TestClient(app2)

timing_log.clear()
resp = client2.get('/hello?name=FastAPI')
print('Response:', resp.json())
print('X-Process-Time header:', resp.headers.get('x-process-time'))

resp2 = client2.get('/slow')
print('Slow response:', resp2.json())
print('X-Process-Time header:', resp2.headers.get('x-process-time'))

print('\nTiming log:')
for entry in timing_log:
    print(f'  {entry["method"]} {entry["path"]} → {entry["status"]} in {entry["ms"]}ms')


**What just happened?**
- The `X-Process-Time` header appears on **every** response without touching any route code.
- Middleware measures wall-clock time around `call_next` — this includes any `await` time inside the route.
- The `timing_log` lets tests assert on middleware behavior without needing a real server.


---
## Step 4 · Verifying middleware behavior with TestClient

TestClient preserves response headers, so we can write assertions against middleware-injected headers. This is the recommended way to test middleware without running a real server.


In [ ]:
# Verify timing header is present and looks correct on multiple endpoints
for path in ['/hello', '/slow']:
    resp = client2.get(path)
    process_time_header = resp.headers.get('x-process-time')
    assert process_time_header is not None, f'Missing header on {path}'
    assert process_time_header.endswith('ms'), f'Unexpected format: {process_time_header}'
    ms_value = float(process_time_header.replace('ms', ''))
    assert ms_value >= 0, 'Process time must be non-negative'
    print(f'{path}: X-Process-Time = {process_time_header}')

# Verify 404 responses also get the header (middleware covers ALL responses)
resp = client2.get('/nonexistent')
assert resp.status_code == 404
assert 'x-process-time' in resp.headers
print(f'/nonexistent: status={resp.status_code}, X-Process-Time={resp.headers["x-process-time"]}')

print('\nAll middleware assertions passed.')


**What just happened?**
- Middleware applies to **every** response including 404s from FastAPI's own routing layer.
- `resp.headers` gives us a case-insensitive dict — `resp.headers.get('x-process-time')` works regardless of the exact case the server sends.
- Testing middleware via response headers is the cleanest approach — no mocking, no monkey-patching.


---
## Step 5 · CORS — `CORSMiddleware`

**CORS (Cross-Origin Resource Sharing)** is a browser security feature. When a JavaScript frontend on `http://localhost:3000` calls your FastAPI on `http://localhost:8000`, the browser first sends a **preflight** `OPTIONS` request to check if the server allows it.

FastAPI's `CORSMiddleware` handles both the preflight and the actual response headers automatically.

| Setting | What it controls |
|---|---|
| `allow_origins` | Which origins can make requests (use specific URLs in production, not `['*']`) |
| `allow_methods` | Which HTTP methods are allowed (`['GET', 'POST', ...]`) |
| `allow_headers` | Which request headers are allowed (e.g. `Authorization`, `Content-Type`) |
| `allow_credentials` | Whether cookies/auth headers are allowed (requires specific origin, not `*`) |


In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app3 = FastAPI(title='CORS Demo')

# Allow the specific frontend origin — never use allow_origins=['*'] in production
# when allow_credentials=True, because browsers block that combination.
app3.add_middleware(
    CORSMiddleware,
    allow_origins=['http://localhost:3000', 'https://myfrontend.example.com'],
    allow_credentials=True,   # allow cookies and Authorization headers
    allow_methods=['GET', 'POST', 'PUT', 'PATCH', 'DELETE', 'OPTIONS'],
    allow_headers=['Content-Type', 'Authorization', 'X-Requested-With'],
)


@app3.get('/data')
def get_data():
    return {'items': [1, 2, 3]}


client3 = TestClient(app3)

# ── Actual request from allowed origin ────────────────────────────────────────
resp = client3.get('/data', headers={'Origin': 'http://localhost:3000'})
print('Status:', resp.status_code)
print('CORS headers on response:')
for key, value in resp.headers.items():
    if 'access-control' in key.lower():
        print(f'  {key}: {value}')


**What just happened?**
- The response includes `Access-Control-Allow-Origin: http://localhost:3000` — the browser sees this and allows the JS code to read the response.
- `Access-Control-Allow-Credentials: true` tells the browser cookies and auth headers are allowed.
- The middleware adds these headers automatically — your route code is unchanged.


---
## Step 6 · Checking CORS preflight and origin rejection

Browsers send an `OPTIONS` **preflight** request before any non-simple request (e.g. one with an `Authorization` header). `CORSMiddleware` handles this automatically.

Requests from disallowed origins should **not** receive the CORS header — the browser then blocks the response.


In [ ]:
# ── Preflight request (OPTIONS) ────────────────────────────────────────────────
preflight_resp = client3.options(
    '/data',
    headers={
        'Origin': 'http://localhost:3000',
        'Access-Control-Request-Method': 'POST',
        'Access-Control-Request-Headers': 'Authorization',
    }
)
print('Preflight status:', preflight_resp.status_code)  # 200 means it's allowed
print('Preflight CORS headers:')
for key, value in preflight_resp.headers.items():
    if 'access-control' in key.lower():
        print(f'  {key}: {value}')

print()

# ── Request from a disallowed origin ──────────────────────────────────────────
bad_origin_resp = client3.get('/data', headers={'Origin': 'http://evil.hacker.com'})
print('Disallowed origin status:', bad_origin_resp.status_code)
allow_origin = bad_origin_resp.headers.get('access-control-allow-origin')
print('Access-Control-Allow-Origin:', allow_origin)
# The header is either absent or set to the allowed origin, not the bad origin
assert allow_origin != 'http://evil.hacker.com', 'Bad origin must NOT be echoed back'
print('Evil origin is correctly blocked: CORS header is NOT the evil origin.')


**What just happened?**
- The preflight `OPTIONS` response confirms `POST` and `Authorization` are allowed from `localhost:3000`.
- The request from `evil.hacker.com` does NOT get `Access-Control-Allow-Origin: http://evil.hacker.com` — so the browser will refuse to expose the response to the JS code.
- **Important:** CORS is a browser enforcement mechanism. The server still processes the request. CORS does not replace server-side authorization.


---
## Step 7 · `GZipMiddleware` — compress large responses

GZip compression reduces the size of large JSON responses, which matters for APIs that return big datasets or long lists. FastAPI's `GZipMiddleware` compresses responses automatically when:
1. The response is larger than `minimum_size` bytes (default: 500)
2. The client sends `Accept-Encoding: gzip` in the request


In [ ]:
import gzip
from fastapi.middleware.gzip import GZipMiddleware

app4 = FastAPI(title='GZip Demo')
app4.add_middleware(GZipMiddleware, minimum_size=500)  # compress responses > 500 bytes


@app4.get('/large')
def large_response():
    """Returns a large JSON payload that benefits from compression."""
    # 1000 records — large enough to compress well
    return {'data': [{'id': i, 'value': f'item-{i}', 'description': 'a' * 50} for i in range(1000)]}


@app4.get('/small')
def small_response():
    """Returns a small payload — below the compression threshold."""
    return {'ok': True}


client4 = TestClient(app4)

# ── Request WITH gzip support ──────────────────────────────────────────────────
resp_compressed = client4.get('/large', headers={'Accept-Encoding': 'gzip'})
content_encoding = resp_compressed.headers.get('content-encoding', 'none')
print('Large response (gzip accepted):')
print('  Content-Encoding:', content_encoding)
print('  Response size (bytes):', len(resp_compressed.content))
print('  Items in data:', len(resp_compressed.json()['data']))  # httpx auto-decompresses

print()

# ── Small payload — NOT compressed (below minimum_size) ───────────────────────
resp_small = client4.get('/small', headers={'Accept-Encoding': 'gzip'})
print('Small response:')
print('  Content-Encoding:', resp_small.headers.get('content-encoding', 'none'))
print('  Body:', resp_small.json())


**What just happened?**
- The large response was compressed — `Content-Encoding: gzip` confirms this.
- `httpx` (which backs `TestClient`) automatically decompresses the response, so `resp.json()` works transparently.
- The small `/small` response has no `Content-Encoding` header because it's under the 500-byte threshold.


---
## Step 8 · Middleware order matters

Middleware executes in **reverse registration order** — the last middleware added is the first to process requests. When combining multiple middlewares, the order affects behavior:

```
Correct order for timing + compression:

app.add_middleware(GZipMiddleware)          ← added first, runs last (compresses body)
app.add_middleware(CORSMiddleware, ...)     ← added second, runs second
app.add_middleware(TimingMiddleware)        ← added last, runs first (times everything including compression)

Request flow:  TimingMiddleware → CORSMiddleware → GZipMiddleware → Route
Response flow: Route → GZipMiddleware → CORSMiddleware → TimingMiddleware
```

The timing middleware should wrap everything — register it last so it runs first.


In [ ]:
# Demonstrate correct middleware stacking order
from fastapi.middleware.cors import CORSMiddleware
from fastapi.middleware.gzip import GZipMiddleware
import time as _time2

app5 = FastAPI(title='Stacked Middleware Demo')

timing_records: list[dict] = []

# Add in this order:
# 1. GZip first (will run last on response — innermost layer)
app5.add_middleware(GZipMiddleware, minimum_size=200)

# 2. CORS second
app5.add_middleware(
    CORSMiddleware,
    allow_origins=['http://localhost:3000'],
    allow_methods=['GET', 'POST'],
    allow_headers=['*'],
)

# 3. Timing last (will run first on request — outermost layer)
@app5.middleware('http')
async def timing_middleware(request: Request, call_next) -> Response:
    t0 = _time2.perf_counter()
    response = await call_next(request)
    ms = round((_time2.perf_counter() - t0) * 1000, 2)
    response.headers['X-Process-Time'] = f'{ms}ms'
    timing_records.append({'path': request.url.path, 'ms': ms})
    return response


@app5.get('/payload')
def get_payload():
    return {'data': ['x' * 100 for _ in range(20)]}  # > 200 bytes, will be compressed


client5 = TestClient(app5)

timing_records.clear()
resp = client5.get(
    '/payload',
    headers={
        'Origin': 'http://localhost:3000',
        'Accept-Encoding': 'gzip',
    }
)

print('Status:', resp.status_code)
print('Content-Encoding:', resp.headers.get('content-encoding', 'none'))
print('CORS header:      ', resp.headers.get('access-control-allow-origin', 'none'))
print('X-Process-Time:   ', resp.headers.get('x-process-time', 'none'))
print('JSON items:       ', len(resp.json()['data']))


**What just happened?**
- All three middleware layers ran: **timing** (X-Process-Time) + **CORS** (Access-Control-Allow-Origin) + **GZip** (Content-Encoding: gzip).
- The timing middleware measured the full end-to-end time including compression overhead.
- `httpx` decompressed the gzip body transparently — `resp.json()` still works.


In [ ]:
# Challenge: Add a middleware to app5 that:
# 1. Injects a custom header: X-API-Version: v1
# 2. Logs the User-Agent header from every request to a list called agent_log
# 3. Test it with TestClient and verify the X-API-Version header is present

# Hint: access request headers with request.headers.get('user-agent')
# Hint: add response headers with response.headers['X-API-Version'] = 'v1'

agent_log: list[str] = []

# Your solution here:

# @app5.middleware('http')
# async def api_version_middleware(request: Request, call_next) -> Response:
#     ...

# Then test:
# resp = client5.get('/payload', headers={'User-Agent': 'TestBrowser/1.0'})
# assert resp.headers.get('x-api-version') == 'v1'
# print('agent_log:', agent_log)


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `BackgroundTasks` | Injected as a route parameter; tasks run **after** the response is sent |
| `add_task(fn, *args)` | Schedules `fn(*args)` — multiple tasks run in order |
| `@app.middleware('http')` | Wraps every request; always `await call_next(request)` |
| Middleware order | Last registered = outermost (runs first); registration order is reversed |
| `CORSMiddleware` | Add specific `allow_origins` — never `['*']` with `allow_credentials=True` |
| Preflight (`OPTIONS`) | Handled automatically by `CORSMiddleware`; no route needed |
| `GZipMiddleware` | Compresses responses above `minimum_size` bytes when client sends `Accept-Encoding: gzip` |
| Testing middleware | Use `resp.headers` in `TestClient` — no real server needed |

> **Tip:** BackgroundTasks is ideal for fire-and-forget work (email, audit log). For truly heavy work, use a task queue like Celery or ARQ instead.

---
## What's next
**Day 10** → WebSockets, Server-Sent Events, and building a real-time notification system with FastAPI.

Mark Day 9 complete in your [tracker](../index.html).
